In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Técnicas de mitigação e supressão de erros

> **Note:** A versão beta de um novo modelo de execução está disponível. O modelo de execução dirigida oferece mais flexibilidade para personalizar seu fluxo de trabalho de mitigação de erros. Consulte o guia [Modelo de execução dirigida](/guides/directed-execution-model) para mais informações.


<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido com os seguintes requisitos.
Recomendamos o uso dessas versões ou mais recentes.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
As técnicas de mitigação e supressão de erros são utilizadas para melhorar a qualidade dos resultados ao escalar para cargas de trabalho maiores. Esta página fornece explicações de alto nível sobre as técnicas de supressão e mitigação de erros disponíveis no Qiskit Runtime.

A célula a seguir importa a primitiva Estimator e cria um backend que será usado para inicializar o Estimator nas células de código posteriores.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Desacoplamento dinâmico
Os circuitos quânticos são executados em hardware IBM&reg; como sequências de pulsos de micro-ondas que precisam ser agendados e executados em intervalos de tempo precisos.
Infelizmente, interações indesejadas entre qubits podem levar a erros coerentes em qubits ociosos. O desacoplamento dinâmico funciona inserindo sequências de pulsos em qubits ociosos para cancelar aproximadamente o efeito desses erros. Cada sequência de pulsos inserida equivale a uma operação identidade, mas a presença física dos pulsos tem o efeito de suprimir erros.
Há muitas escolhas possíveis de sequências de pulsos, e qual sequência é melhor para cada caso particular permanece uma [área ativa de pesquisa](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Observe que o desacoplamento dinâmico é principalmente útil para circuitos que contêm lacunas nas quais alguns qubits ficam ociosos sem nenhuma operação agindo sobre eles. Se as operações no circuito estiverem muito densamente agrupadas, de modo que todos os qubits estejam ocupados na maior parte do tempo, a adição de pulsos de desacoplamento dinâmico pode não melhorar o desempenho. Na verdade, pode até piorar o desempenho devido a imperfeições nos próprios pulsos.

O diagrama abaixo representa o desacoplamento dinâmico com uma sequência de pulsos XX. O circuito abstrato à esquerda é mapeado para um cronograma de pulsos de micro-ondas no canto superior direito. O canto inferior direito representa o mesmo cronograma, mas com uma sequência de dois pulsos X inserida durante um período ocioso do primeiro qubit.

![Representação do desacoplamento dinâmico](../docs/images/guides/error-mitigation-explanation/dd.avif)

O desacoplamento dinâmico pode ser habilitado definindo `enable` como `True` nas [opções de desacoplamento dinâmico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). A opção `sequence_type` pode ser usada para escolher entre várias sequências de pulsos diferentes. O tipo de sequência padrão é `"XX"`.

A célula de código a seguir mostra como habilitar o desacoplamento dinâmico para o Estimator e escolher uma sequência de desacoplamento dinâmico.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Twirling de Pauli
O twirling, também conhecido como [compilação randomizada](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), é uma técnica amplamente utilizada para converter canais de ruído arbitrários em canais de ruído com estrutura mais específica.

O twirling de Pauli é um tipo especial de twirling que utiliza operações de Pauli. Ele tem o efeito de transformar qualquer canal quântico em um canal de Pauli. Realizado isoladamente, pode mitigar o ruído coerente, pois o ruído coerente tende a se acumular quadraticamente com o número de operações, enquanto o ruído de Pauli se acumula linearmente. O twirling de Pauli é frequentemente combinado com outras técnicas de mitigação de erros que funcionam melhor com ruído de Pauli do que com ruído arbitrário.

O twirling de Pauli é implementado envolvendo um conjunto escolhido de portas com portas de Pauli de qubit único escolhidas aleatoriamente, de tal forma que o efeito ideal da porta permaneça o mesmo. O resultado é que um único circuito é substituído por um conjunto aleatório de circuitos, todos com o mesmo efeito ideal. Ao amostrar o circuito, as amostras são retiradas de múltiplas instâncias aleatórias, em vez de apenas uma.

![Representação do twirling de Pauli](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Como a maioria dos erros no hardware quântico atual provém de portas de dois qubits, essa técnica é frequentemente aplicada exclusivamente a portas de dois qubits (nativas). O diagrama a seguir representa alguns twirls de Pauli para as portas CNOT e ECR. Cada circuito dentro de uma linha tem o mesmo efeito ideal.

![Representação de twirls de portas](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

O twirling de Pauli pode ser habilitado definindo `enable_gates` como `True` nas [opções de twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Outras opções notáveis incluem:

- `num_randomizations`: O número de instâncias de circuito a serem retiradas do conjunto de circuitos com twirling.
- `shots_per_randomization`: O número de shots a serem amostrados de cada instância de circuito.

A célula de código a seguir mostra como habilitar o twirling de Pauli e definir essas opções para o estimator. Nenhuma dessas opções precisa ser definida explicitamente.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Extinção de erros de leitura por twirling (TREX)
A [extinção de erros de leitura por twirling (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) mitiga o efeito de erros de medição na estimativa de valores esperados de observáveis de Pauli.
Ela se baseia na noção de medições com twirling, que são realizadas substituindo aleatoriamente portas de medição por uma sequência de (1) uma porta X de Pauli, (2) uma medição e (3) inversão de bit clássico. Assim como no twirling de portas padrão, essa sequência é equivalente a uma medição simples na ausência de ruído, conforme representado no diagrama a seguir:

![Representação do twirling de medição](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

Na presença de erros de leitura, o twirling de medição tem o efeito de diagonalizar a matriz de transferência de erros de leitura, tornando-a fácil de inverter. Estimar a matriz de transferência de erros de leitura requer a execução de circuitos de calibração adicionais, o que introduz uma pequena sobrecarga.

O TREX pode ser habilitado definindo `measure_mitigation` como `True` nas [opções de resiliência do Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para o Estimator. As opções para aprendizado de ruído de medição são descritas [aqui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Assim como no twirling de portas, você pode definir o número de randomizações de circuito e o número de shots por randomização.

A célula de código a seguir mostra como habilitar o TREX e definir essas opções para o estimator. Nenhuma dessas opções precisa ser definida explicitamente.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Extrapolação de ruído zero (ZNE)
A extrapolação de ruído zero (ZNE) é uma técnica para mitigar erros na estimativa de valores esperados de observáveis. Embora frequentemente melhore os resultados, não há garantia de que produza um resultado não viesado.

A ZNE consiste em duas etapas:

1. _Amplificação de ruído_: O circuito quântico original é executado múltiplas vezes em diferentes taxas de ruído.
2. _Extrapolação_: O resultado ideal é estimado extrapolando os resultados de valor esperado com ruído até o limite de ruído zero.

Tanto a etapa de amplificação de ruído quanto a de extrapolação podem ser implementadas de muitas formas diferentes. O Qiskit Runtime implementa a amplificação de ruído por "dobramento digital de portas", o que significa que as portas de dois qubits são substituídas por sequências equivalentes da porta e seu inverso. Por exemplo, substituir um unitário $U$ por $U U^\dagger U$ resultaria em um fator de amplificação de ruído de 3. Para a extrapolação, você pode escolher entre várias formas funcionais, incluindo um ajuste linear ou exponencial.
A imagem abaixo representa o dobramento digital de portas à esquerda e o procedimento de extrapolação à direita.

![Representação da ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

A ZNE pode ser habilitada definindo `zne_mitigation` como `True` nas [opções de resiliência do Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para o Estimator.
As opções do Qiskit Runtime para ZNE são descritas [aqui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). As seguintes opções são notáveis:

- `noise_factors`: Os fatores de ruído a serem usados para amplificação de ruído.
- `extrapolator`: A forma funcional a ser usada para a extrapolação.

A célula de código a seguir mostra como habilitar a ZNE e definir essas opções para o estimator. Nenhuma dessas opções precisa ser definida explicitamente.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplificação Probabilística de Erros (PEA)
Um dos principais desafios na ZNE é amplificar com precisão o ruído que afeta o circuito alvo. O dobramento de portas fornece uma maneira fácil de realizar essa amplificação, mas pode ser impreciso e levar a resultados incorretos. Veja o artigo ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), e especificamente a página 4 das informações suplementares para detalhes. A amplificação probabilística de erros fornece uma abordagem mais precisa para a amplificação de erros por meio do aprendizado de ruído.

A PEA é uma técnica mais sofisticada que realiza experimentos preliminares para reconstruir o ruído e, em seguida, usa essas informações para realizar uma amplificação precisa. Ela começa aprendendo o modelo de ruído com twirling de cada camada de portas de entrelaçamento no circuito antes de serem executadas (veja [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) para opções de aprendizado relevantes). Após a fase de aprendizado, os circuitos são executados em cada fator de ruído, onde cada camada de entrelaçamento dos circuitos é amplificada por injeção probabilística de ruído de qubit único proporcional ao modelo de ruído aprendido correspondente. Veja o artigo ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) para mais detalhes.

A PEA consiste em três etapas:
1. _Aprendizado_: O modelo de ruído com twirling de cada camada de portas de entrelaçamento no circuito é aprendido.
1. _Amplificação de ruído_: O circuito quântico original é executado múltiplas vezes em diferentes fatores de ruído.
2. _Extrapolação_: O resultado ideal é estimado extrapolando os resultados de valor esperado com ruído até o limite de ruído zero.

Para experimentos em escala de utilidade, a PEA é frequentemente a melhor escolha.

Como a PEA é uma técnica de amplificação de ruído ZNE, você também precisa habilitar a ZNE definindo `resilience.zne_mitigation = True`. Outras opções de [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) podem ser usadas adicionalmente para definir extrapoladores, níveis de amplificação e assim por diante. A PEA requer um modelo de ruído, que é gerado automaticamente ao usar primitivas.

O trecho a seguir fornece um exemplo em que a PEA é usada para mitigar o resultado de um job do Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Cancelamento probabilístico de erros (PEC)
O cancelamento probabilístico de erros (PEC) é uma técnica para mitigar erros na estimativa de valores esperados de observáveis. Ao contrário da ZNE, ele retorna uma estimativa não viesada do valor esperado. No entanto, geralmente incorre em uma sobrecarga maior.

No PEC, o efeito de um circuito alvo ideal é expresso como uma combinação linear de circuitos com ruído que são realmente implementáveis na prática:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

A saída do circuito ideal pode então ser reproduzida executando diferentes instâncias de circuito com ruído retiradas de um conjunto aleatório definido pela combinação linear. Se os coeficientes $\eta_i$ formarem uma distribuição de probabilidade, eles podem ser usados diretamente como as probabilidades do conjunto. Na prática, alguns dos coeficientes são negativos, portanto formam uma distribuição de quasi-probabilidade. Eles ainda podem ser usados para definir um conjunto aleatório, mas há uma sobrecarga de amostragem relacionada à negatividade da distribuição de quasi-probabilidade, que é caracterizada pela quantidade

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

A sobrecarga de amostragem é um fator multiplicativo no número de shots necessários para estimar um valor esperado com uma determinada precisão, em comparação com o número de shots que seriam necessários no circuito ideal. Ela escala quadraticamente com $\gamma$, que por sua vez escala exponencialmente com a profundidade do circuito.

O PEC pode ser habilitado definindo `pec_mitigation` como `True` nas [opções de resiliência do Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) para o Estimator.
As opções do Qiskit Runtime para PEC são descritas [aqui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Um limite para a sobrecarga de amostragem pode ser definido usando a opção `max_overhead`. Observe que limitar a sobrecarga de amostragem pode fazer com que a precisão do resultado exceda a precisão solicitada. O valor padrão de `max_overhead` é 100.

A célula de código a seguir mostra como habilitar o PEC e definir a opção `max_overhead` para o estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Próximos passos
- Confira o [tutorial](/tutorials/combine-error-mitigation-techniques) sobre como combinar opções de mitigação de erros com a primitiva Estimator.
- [Configurar mitigação de erros.](configure-error-mitigation)
- [Configurar supressão de erros.](configure-error-suppression)
- Explore outras [opções](runtime-options-overview) para as primitivas do Qiskit Runtime.
- Decida em qual [modo de execução](execution-modes) executar seu job.